# Text-to-SQL with AgentCore and Postgres MCP

This notebook demonstrates querying a PostgreSQL analytics database through two Postgres MCP servers running on AWS AgentCore.

**Infrastructure used:**
- **AgentCore runtime** — stack `demo-postgres-mcp-agent` (deployed via `deploy_agentcore_agent_cf.sh`)
- **Views MCP server** — stack `demo-postgres-mcp-views` — tenant mode, scoped to allowed views/tenants (deployed via `deploy_views_mcp_cf.sh`)
- **Admin MCP server** — stack `demo-postgres-mcp-admin` — admin mode, full database access, no tenant restrictions (deployed via `deploy_admin_mcp_cf.sh`)

## Dependencies

In [ ]:
%pip install --quiet boto3
print("✅ Dependencies installed")

## Imports

In [15]:
import boto3
import json
import uuid
import time
from typing import Dict, Any

print("✅ Imports loaded")

✅ Imports loaded


## Configuration

Pulls the AgentCore runtime ARN from the `demo-postgres-mcp-agent` stack and both MCP endpoints + API keys from their respective stacks automatically.

In [16]:
AWS_PROFILE = "sandbox"
AWS_REGION  = "us-east-1"

AGENTCORE_STACK_NAME  = "demo-postgres-mcp-agent"
VIEWS_MCP_STACK_NAME  = "demo-postgres-mcp-views"
ADMIN_MCP_STACK_NAME  = "demo-postgres-mcp-admin"

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
cf      = session.client("cloudformation", region_name=AWS_REGION)
sm      = session.client("secretsmanager", region_name=AWS_REGION)

def _stack_output(stack_name: str, key: str) -> str:
    outputs = cf.describe_stacks(StackName=stack_name)["Stacks"][0]["Outputs"]
    return next(o["OutputValue"] for o in outputs if o["OutputKey"] == key)

# AgentCore runtime
AGENTCORE_RUNTIME_ARN = _stack_output(AGENTCORE_STACK_NAME, "AgentRuntimeArn")

# Views MCP (tenant mode)
VIEWS_MCP_URL     = _stack_output(VIEWS_MCP_STACK_NAME, "McpEndpointUrl")
VIEWS_MCP_API_KEY = sm.get_secret_value(
    SecretId=_stack_output(VIEWS_MCP_STACK_NAME, "ApiKeySecretArn")
)["SecretString"]

# Admin MCP (unrestricted)
ADMIN_MCP_URL     = _stack_output(ADMIN_MCP_STACK_NAME, "McpEndpointUrl")
ADMIN_MCP_API_KEY = sm.get_secret_value(
    SecretId=_stack_output(ADMIN_MCP_STACK_NAME, "ApiKeySecretArn")
)["SecretString"]

DEFAULT_MODEL_CONFIG = {
    "model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "region_name": AWS_REGION,
    "temperature": 0,
    "max_tokens": 4096,
}

print(f"✅ AgentCore runtime : {AGENTCORE_RUNTIME_ARN}")
print(f"✅ Views MCP         : {VIEWS_MCP_URL}")
print(f"✅ Admin MCP         : {ADMIN_MCP_URL}")
print(f"✅ API keys loaded from Secrets Manager")

✅ AgentCore runtime : arn:aws:bedrock-agentcore:us-east-1:008701887645:runtime/demo_mcp_agent_dev_f951a160-5bGXTZB0n0
✅ Views MCP         : http://demo-p-LoadB-D2hlO3hHWWzi-1205460695.us-east-1.elb.amazonaws.com/mcp
✅ Admin MCP         : http://demo-p-LoadB-2gOh0tisjkJc-455545092.us-east-1.elb.amazonaws.com/mcp
✅ API keys loaded from Secrets Manager


## Helper Functions

In [17]:
def make_session_id() -> str:
    return f"session_{uuid.uuid4().hex}_{int(time.time())}"


def views_mcp_config(tenant_id: str) -> Dict:
    """MCP config for the Views server — tenant-scoped, read-only views."""
    return {
        "postgres": {
            "transport": "streamable_http",
            "url": VIEWS_MCP_URL,
            "headers": {
                "x-api-key": VIEWS_MCP_API_KEY,
                "x-tenant-id": tenant_id,
            },
        }
    }


def admin_mcp_config() -> Dict:
    """MCP config for the Admin server — full database access, no tenant restrictions."""
    return {
        "postgres": {
            "transport": "streamable_http",
            "url": ADMIN_MCP_URL,
            "headers": {
                "x-api-key": ADMIN_MCP_API_KEY,
            },
        }
    }


def invoke_text_to_sql(
    client,
    query: str,
    mcp_config: Dict,
    session_id: str = None,
    tenant_id: str = None,
    actor_id: str = "user-1",
    user_id: str = "demo",
    model_config: Dict = None,
) -> Dict[str, Any]:
    """Invoke the AgentCore runtime with a text-to-SQL query via the Postgres MCP tool."""
    session_id   = session_id or make_session_id()
    model_config = model_config or DEFAULT_MODEL_CONFIG

    payload = {
        "query":        query,
        "model_config": model_config,
        "mcp_config":   mcp_config,
        "session_id":   session_id,
        "actor_id":     actor_id,
        "user_id":      user_id,
    }
    if tenant_id:
        payload["tenant_id"] = tenant_id

    label = f"[{tenant_id}] " if tenant_id else "[admin] "
    print(f"📤 {label}{query}")
    print(f"   session: {session_id}")

    try:
        t0   = time.time()
        ttft = None

        response = client.invoke_agent_runtime(
            agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
            runtimeSessionId=session_id,
            payload=json.dumps(payload).encode("utf-8"),
        )

        content = []
        print("\n🤖 Response:")
        print("-" * 80)
        for raw in response["response"].iter_lines(chunk_size=1):
            if not raw:
                continue
            if ttft is None:
                ttft = time.time() - t0  # first non-empty chunk

            line = raw.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                try:
                    line = json.loads(line)
                except Exception:
                    pass
                if isinstance(line, str) and line.startswith("data: "):
                    line = line[6:].rstrip("\n")
            print(line, flush=True)
            content.append(line)
        print("-" * 80)

        final = ""
        for line in content:
            try:
                d = json.loads(line) if isinstance(line, str) else line
                if d.get("type") == "final" and "response" in d:
                    final = d["response"]
                    break
            except Exception:
                pass
        if not final:
            for line in content:
                try:
                    d = json.loads(line) if isinstance(line, str) else line
                    if d.get("type") == "content_delta":
                        final += d.get("content", "")
                except Exception:
                    pass

        duration = time.time() - t0
        print(f"\n⏱  TTFT: {ttft:.2f}s  |  Total: {duration:.2f}s")
        print(f"\n📋 Answer:\n{final}\n")
        return {
            "success": True,
            "answer": final,
            "ttft": ttft,
            "duration": duration,
            "session_id": session_id,
        }

    except Exception as e:
        import traceback
        print(f"❌ Error: {e}")
        traceback.print_exc()
        return {"success": False, "answer": str(e), "ttft": None, "duration": 0, "session_id": session_id}


print("✅ Helper functions loaded")

✅ Helper functions loaded


## Initialize Client

In [18]:
client = session.client("bedrock-agentcore", region_name=AWS_REGION)
print(f"✅ bedrock-agentcore client ready (profile={AWS_PROFILE}, region={AWS_REGION})")

✅ bedrock-agentcore client ready (profile=sandbox, region=us-east-1)


---
## Views MCP Examples

The Views MCP runs in `tenant` mode — requests are scoped by `x-tenant-id` header and restricted to pre-approved views and tenant IDs.

### Example V1 — Tenant-Level KPI Query (tenant-a, expected to succeed)

In [19]:
result_v1 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for me",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [tenant-a] Use the Postgres MCP tool to get the order count and gross revenue KPI for me
   session: session_87db12876c0240ee9b082e117f7955a9_1780339094

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help"}
{"type": "content_delta", "content": " you get the order count and gross revenue"}
{"type": "content_delta", "content": " KPI."}
{"type": "content_delta", "content": " First"}
{"type": "content_delta", "content": ", let me check"}
{"type": "content_delta", "content": " what relations"}
{"type": "content_delta", "content": " are available to"}
{"type": "content_delta", "content": " work"}
{"type": "content_delta", "content": " with."}
{"type": "tool_use_start", "tool": "list_allowed_relations", "tool_use_id": "tooluse_NcXTah832bGj6VZhDcOP7d", "input": {}}
{"type": "tool_use_result", "tool": "list_allowed_relations", "tool_use_id": "tooluse_NcXTah832bGj6VZhDcOP7d", "output": "[{'relation_name':

### Example V2 — Non-Allowed Tenant Query (tenant-z, expected to fail)

`tenant-z` is not in the server's `TENANT_ALLOWED_TENANT_IDS` list — the MCP server should reject the request.

In [20]:
result_v2 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for this tenant.",
    mcp_config=views_mcp_config("tenant-z"),
    tenant_id="tenant-z",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)
print(f"⚠️  Expected failure — success={result_v2['success']}")

📤 [tenant-z] Use the Postgres MCP tool to get the order count and gross revenue KPI for this tenant.
   session: session_932527dd83454120ae813debfd71bf73_1780339114

🤖 Response:
--------------------------------------------------------------------------------
{"type": "error", "message": "Failed to load tool <strands.tools.mcp.mcp_client.MCPClient object at 0xffff9d3a3050>: Failed to start MCP client: the client initialization failed", "error_type": "ValueError"}
--------------------------------------------------------------------------------

⏱  TTFT: 0.73s  |  Total: 0.73s

📋 Answer:


⚠️  Expected failure — success=True


### Example V3 — List Accessible Tables (Views MCP)

Ask the Views MCP what tables and views it can see — should return only the pre-approved analytics views.

In [21]:
result_v3 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to list all tables and views you have access to. Provide the exact view/tables names in your answer.",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [tenant-a] Use the Postgres MCP tool to list all tables and views you have access to. Provide the exact view/tables names in your answer.
   session: session_e4df7ea8607b4617b46a6383f2c6ac73_1780339117

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll list"}
{"type": "content_delta", "content": " all the"}
{"type": "content_delta", "content": " tables and views available to"}
{"type": "content_delta", "content": " you"}
{"type": "content_delta", "content": "."}
{"type": "tool_use_start", "tool": "list_allowed_relations", "tool_use_id": "tooluse_8myrQPh5FO2w9Nu8W2Nq3C", "input": {}}
{"type": "tool_use_result", "tool": "list_allowed_relations", "tool_use_id": "tooluse_8myrQPh5FO2w9Nu8W2Nq3C", "output": "[{'relation_name': 'analytics_app.customer_revenue_summary', 'relation_type': 'VIEW', 'columns': [{'column_name': 'tenant_id', 'data_type': 'text'}, {'column_name': 'customer_id', 'data_type': 'intege

### Example V4 — Write Operations: INSERT and DELETE (expected to fail)

The Views MCP connects with a read-only database user — INSERT and DELETE should be rejected at the database level.

In [22]:
result_v4_insert = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to insert a test row into analytics_app.executive_kpis with all columns set to 0 or empty string.",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)
print(f"⚠️  INSERT — expected failure — success={result_v4_insert['success']}")

result_v4_delete = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to delete all rows from analytics_app.executive_kpis.",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)
print(f"⚠️  DELETE — expected failure — success={result_v4_delete['success']}")

📤 [tenant-a] Use the Postgres MCP tool to insert a test row into analytics_app.executive_kpis with all columns set to 0 or empty string.
   session: session_959a4a6da904401ca84eaed1118a064f_1780339123

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I appreciate"}
{"type": "content_delta", "content": " your request, but I need to clar"}
{"type": "content_delta", "content": "ify the"}
{"type": "content_delta", "content": " capabilities"}
{"type": "content_delta", "content": " available"}
{"type": "content_delta", "content": " to me. The tools"}
{"type": "content_delta", "content": " I have access to are **"}
{"type": "content_delta", "content": "rea"}
{"type": "content_delta", "content": "d-only** SQL"}
{"type": "content_delta", "content": " tools designe"}
{"type": "content_delta", "content": "d for que"}
{"type": "content_delta", "content": "rying tenant"}
{"type": "content_delta", "content": "-"}
{"typ

---
## Admin MCP Examples

The Admin MCP runs in `admin` mode — no tenant restrictions, full database access.

### Example A1 — List All Accessible Tables (Admin MCP)

Should return every table and view in the database, across all schemas.

In [23]:
result_a1 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to list all tables and views available in the database, grouped by schema. Provide the exact view/tables names in your answer.",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to list all tables and views available in the database, grouped by schema. Provide the exact view/tables names in your answer.
   session: session_27471b3b7c574e5ab6e7696eaa177962_1780339133

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help"}
{"type": "content_delta", "content": " you list all tables and views in"}
{"type": "content_delta", "content": " the database,"}
{"type": "content_delta", "content": " groupe"}
{"type": "content_delta", "content": "d by schema. Let me start"}
{"type": "content_delta", "content": " by getting"}
{"type": "content_delta", "content": " the list of schemas"}
{"type": "content_delta", "content": "."}
{"type": "tool_use_start", "tool": "list_schemas", "tool_use_id": "tooluse_3pwkIlMdSsrcVmF9LeNhkN", "input": {}}
{"type": "tool_use_result", "tool": "list_schemas", "tool_use_id": "tooluse_3pwkIlMdSsrcVmF9LeNhkN", "output": "[{'sch

---
## Views vs Raw — Speed Comparison

The Views path should be significantly faster end-to-end: less schema for the model to reason over, simpler SQL to generate, and a lighter database query to execute.

### P1 — Views 

In [26]:
result_p1_views = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [tenant-a] Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a
   session: session_da9cedc6110c410c8ab279c20c797a35_1780339433

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help"}
{"type": "content_delta", "content": " you get the order count and gross revenue"}
{"type": "content_delta", "content": " KPI for tenant a"}
{"type": "content_delta", "content": "."}
{"type": "content_delta", "content": " Let me start"}
{"type": "content_delta", "content": " by exploring"}
{"type": "content_delta", "content": " the available relations"}
{"type": "content_delta", "content": "."}
{"type": "tool_use_start", "tool": "list_allowed_relations", "tool_use_id": "tooluse_uzIaCA1N5uevy47SyAFHxx", "input": {}}
{"type": "tool_use_result", "tool": "list_allowed_relations", "tool_use_id": "tooluse_uzIaCA1N5uevy47SyAFHxx", "output": "[{'relation_name': 'analytics_app.customer_revenue_s

### P2 — Same Query via Admin MCP (raw tables)

In [27]:
result_p2_admin = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a
   session: session_5a93677246854461a38a69de22e7063f_1780339443

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help"}
{"type": "content_delta", "content": " you get the order count and gross revenue"}
{"type": "content_delta", "content": " KPI for tenant a"}
{"type": "content_delta", "content": "."}
{"type": "content_delta", "content": " Let me first"}
{"type": "content_delta", "content": " explore"}
{"type": "content_delta", "content": " the database"}
{"type": "content_delta", "content": " structure to understand what"}
{"type": "content_delta", "content": " tables"}
{"type": "content_delta", "content": " are"}
{"type": "content_delta", "content": " available."}
{"type": "tool_use_start", "tool": "list_schemas", "tool_use_id": "tooluse_1BYLKTDpyLzyMPAmtqgWca", "input": {}}
{"type": "tool_use_result", "t

### Performance Comparison

In [28]:
views_ttft  = result_p1_views["ttft"]
views_total = result_p1_views["duration"]
admin_ttft  = result_p2_admin["ttft"]
admin_total = result_p2_admin["duration"]

print(f"{"":30} {"TTFT":>8} {"Total":>8}")
print("-" * 48)
print(f"{"P1 Views MCP (1 view)":30} {views_ttft:>7.2f}s {views_total:>7.2f}s")
print(f"{"P2 Admin MCP (7 raw tables)":30} {admin_ttft:>7.2f}s {admin_total:>7.2f}s")
print()
if admin_total and views_total:
    print(f"Admin was {admin_total / views_total:.1f}x slower end-to-end")

                                   TTFT    Total
------------------------------------------------
P1 Views MCP (1 view)             1.75s   10.57s
P2 Admin MCP (7 raw tables)       2.47s   19.04s

Admin was 1.8x slower end-to-end
